In [3]:
from scipy import integrate, stats, optimize
import numpy as np

mu, sd, x, T = 1e-3, 5e-4, 1, 1500     # change T to 15000 if using total time on test
post_un = lambda l: (l*T)**x*np.exp(-l*T)*stats.norm.pdf(l, mu, sd)
Z = integrate.quad(post_un, 0, 5e-3)[0]
post = lambda l: post_un(l)/Z

mean = integrate.quad(lambda l: l*post(l), 0, 5e-3)[0]
var  = integrate.quad(lambda l: l*l*post(l), 0, 5e-3)[0] - mean**2
cdf  = lambda L: integrate.quad(post, 0, L)[0]
pct  = lambda p: optimize.brentq(lambda L: cdf(L)-p, 1e-7, 5e-3)
print(mean, np.sqrt(var), pct(0.05), pct(0.95))   # mean, sd, 90% lower limit

0.0009688256406119194 0.00040852527886943966 0.0003359405363261159 0.0016793944272470765


Default to **Gamma**. Whenever a problem hands you a rate as *mean + standard deviation* with Poisson/exponential data, moment-match to a Gamma (α0=(μ/σ)2\alpha_0=(\mu/\sigma)^2
α0​=(μ/σ)2, β0=μ/σ2\beta_0=\mu/\sigma^2
β0​=μ/σ2), update by adding failures to α\alpha
α and time to β\beta
β, and read percentiles from χ2α′2\chi^2_{2\alpha'}
χ2α′2​. Two minutes, no computer, no integral. Only fall back to numerical integration if the problem *explicitly* names a Normal (or other non-conjugate) prior — and now you can do that in Python too.

## 3.27 Lognormal problem

Rebuilding the context

$$
\lambda ∼ N(\mu, \sigma^2)
$$

Quick facts about lognormal:
median is:
$$
\lambda_50 = \sqrt{\lambda_{05} \lambda_{95}} = \sqrt{(3\times 10^{-4})(3 \times 10^{-2})} = 3 \times 10^{-3}
$$

Therefore, 

$$
\mu = ln(3 \times 10^{-3}) = -5.809
$$

Error factor

$$
EF = \frac{\lambda_{95}}{\lambda_{50}} = \frac{\lambda_{50}}{\lambda_{05}} = \frac{3\times 10^{-2}}{3\times 10^{-3}} = 10
$$

$\sigma$ is given as:

$$
EF = e^{1.645\sigma},
$$

$$
\sigma = \frac{ln(EF)}{1.645} = \frac{ln(10)}{1.645} = 1.40
$$

In [ ]:
from scipy import integrate, stats, optimize
import numpy as np

# Problem 3.27 from Modarres
# Lower and Upper limits of the lognormal distribution
LO, HI = 3e-4, 3e-2
mu = np.log(np.sqrt(LO*HI))
EF = np.sqrt(HI/LO)
sigma = np.log(EF)/1.645

# Input data about diesel engine failures, twice in one year, 8760 hours
x, T = 2, 8760
prior = lambda l: stats.lognorm.pdf(l, s=sigma, scale=np.exp(mu))
like = lambda l: (l*T)**x * np.exp(-l*T)
un = lambda l: like(l)*prior(l)

LO, HI = 1e-7, 1e-1
Z = integrate.quad(un, LO, HI)[0]
post = lambda l: un(l)/Z

mean = integrate.quad(lambda l: l*post(l), LO, HI)[0]
var  = integrate.quad(lambda l: l*l*post(l), LO, HI)[0] - mean**2
cdf  = lambda L: integrate.quad(post, LO, L)[0]
pct  = lambda p: optimize.brentq(lambda L: cdf(L)-p, LO, HI)
# print(mean, np.sqrt(var), pct(0.05), pct(0.95))   # mean, sd, 90% lower limit
# Confidence interval for the failure rate post, 5 decimals
print(f"Mean: {mean:.5e}, SD: {np.sqrt(var):.5e}")
print(f"90% confidence interval: [{pct(0.05):.5e}, {pct(0.95):.5e}]")

0.0003599629571142224 0.00018831059620075702 0.00011934797233176414 0.0007165407584309128
90% confidence interval: [0.00011934797233176414, 0.0007165407584309128]


# Bayesian Reliability — Exam Cheatsheet

> **The one idea.** Every problem is the same machine:
> $$\pi(\theta\mid\text{data}) = \frac{L(\text{data}\mid\theta)\,\pi_0(\theta)}{\displaystyle\int L(\text{data}\mid\theta)\,\pi_0(\theta)\,d\theta} \;\propto\; \underbrace{L(\text{data}\mid\theta)}_{\text{data model}}\cdot\underbrace{\pi_0(\theta)}_{\text{prior on parameter}}$$
> $\theta$ is the **parameter** you want (a rate $\lambda$, a probability $p$). The variation between problems is only: (a) how the prior is *summarized*, (b) what the *data model* is, (c) whether the pair is *conjugate* (closed form) or not (integrate).

---

## The universal recipe (4 steps)

1. **Identify the parameter $\theta$ and the data model (likelihood).**
   The likelihood is *dictated by physics*, not chosen. Counts in time → Poisson. Times-to-failure → exponential. Failures-in-demands → binomial. The prior lives on $\theta$ (parameter-space); the likelihood lives on the data (data-space). **Never conflate the two.**

2. **Decode the prior $\pi_0(\theta)$ from its given summary** (mean+SD, or 90% bounds/EF, or stated parameters). See *Decode table*.

3. **Check conjugacy.**
   - *Conjugate* (Gamma↔Poisson/exp, Beta↔binomial): write posterior parameters **by inspection** — no integral. See *Conjugate shortcuts*.
   - *Not conjugate* (lognormal, normal prior): **integrate numerically**. See *Python template*.

4. **Extract what's asked.**
   - Point estimate: posterior **mean** $E[\theta]$, **variance** $\mathrm{Var}[\theta]$.
   - Bounds: **percentiles** of the posterior. Mind one-sided vs two-sided (see *Interval mechanics*).

---

## Decode table — summary → prior parameters

| Given as… | Prior | Recover parameters |
|---|---|---|
| **mean $\mu$, std $\sigma$** of a rate | **Gamma**$(\alpha_0,\beta_0)$ | $\alpha_0=\left(\dfrac{\mu}{\sigma}\right)^2,\quad \beta_0=\dfrac{\mu}{\sigma^2}$ |
| **mean $\mu$, std $\sigma$** of a probability | **Beta**$(a_0,b_0)$ | $v=\dfrac{\mu(1-\mu)}{\sigma^2}-1,\quad a_0=\mu v,\ b_0=(1-\mu)v$  *(needs $\sigma^2<\mu(1-\mu)$)* |
| **90% bounds** $\lambda_{05},\lambda_{95}$ (lognormal) | **Lognormal**$(\mu,\sigma)$ | median $=\sqrt{\lambda_{05}\lambda_{95}}$, $\ \mathrm{EF}=\sqrt{\lambda_{95}/\lambda_{05}}=\dfrac{\lambda_{95}}{\text{median}}$, $\ \mu=\ln(\text{median}),\ \sigma=\dfrac{\ln \mathrm{EF}}{1.645}$ |
| **mean $\mu$, std $\sigma$** (lognormal form) | **Lognormal**$(\mu_L,\sigma_L)$ | $\sigma_L^2=\ln\!\big(1+(\sigma/\mu)^2\big),\quad \mu_L=\ln\mu-\sigma_L^2/2$ |
| **mean & std**, told "normal" | **Normal**$(\mu,\sigma)$ | use directly |
| **a range** ("more than $a$", "between $a$ and $b$") + "uniform/noninformative" | **Uniform on $[a,b]$** | height $\pi_0=\dfrac{1}{b-a}$ (constant); for "more than $a$" the upper edge is the natural max ($b=1$ for a reliability/probability). **The range *is* the prior — it is not discarded.** |

> **EF note:** $\mathrm{EF}=e^{1.645\,\sigma}$ holds for **90% bounds (5th/95th)**. For **95% bounds (2.5/97.5)** swap $1.645\to1.960$.

---

## Likelihood models — data → kernel in the parameter

| Data scenario | Distribution | Likelihood kernel (drop constants) |
|---|---|---|
| $x$ failures in total time $T$ | Poisson$(\lambda T)$ | $L(\lambda)\propto \lambda^{x}\,e^{-\lambda T}$ |
| $n$ observed failure times, total time $T=\sum t_i$ | Exponential$(\lambda)$ | $L(\lambda)\propto \lambda^{n}\,e^{-\lambda T}$  *(same kernel as Poisson with $x=n$)* |
| $k$ failures in $n$ demands | Binomial$(n,p)$ | $L(p)\propto p^{k}(1-p)^{n-k}$ |
| measurements $x_i$, known meas. $\sigma$ | Normal | $L(\mu)\propto \exp\!\big[-\tfrac{1}{2\sigma^2}\sum(x_i-\mu)^2\big]$ |
| times-to-failure, **shape $\beta$ known** | Weibull$(\beta,\eta)$ | with $\lambda\equiv\eta^{-\beta}$: $L(\lambda)\propto \lambda^{n}e^{-\lambda\sum t_i^{\beta}}$ → **Gamma-conjugate in $\lambda$** |

> **Key collapse:** exponential-TTF and Poisson-counts give the *identical* kernel $\lambda^{\,r}e^{-\lambda T}$ (with $r$ = number of failures, $T$ = total time on test). That's why the same Gamma machinery covers both.
>
> **Total time on test:** $T=(\text{number of units})\times(\text{observation time})$. *e.g.* 10 units × 1500 h $=15000$ component-hours. Watch for this factor.

---

## Conjugate shortcuts (memorize — no integral needed)

### Gamma–Poisson / Gamma–Exponential  (rate $\lambda$)
- Prior $\lambda\sim\text{Gamma}(\alpha_0,\beta_0)$, with $E[\lambda]=\dfrac{\alpha_0}{\beta_0}$, $\mathrm{Var}=\dfrac{\alpha_0}{\beta_0^2}$.
- Data: $r$ failures, total time $T$.
- **Posterior:** $\boxed{\text{Gamma}(\alpha_0+r,\ \beta_0+T)}$ — *add failures to shape, add time to rate.*
- Posterior mean $=\dfrac{\alpha'}{\beta'}$, variance $=\dfrac{\alpha'}{\beta'^2}$.

### Beta–Binomial  (probability $p$)
- Prior $p\sim\text{Beta}(a_0,b_0)$, with $E[p]=\dfrac{a_0}{a_0+b_0}$.
- Data: $k$ failures in $n$ demands.
- **Posterior:** $\boxed{\text{Beta}(a_0+k,\ b_0+n-k)}$ — *add failures to $a$, successes to $b$.*
- Posterior mean $=\dfrac{a_0+k}{a_0+b_0+n}$.

> **No conjugate form:** lognormal prior, normal prior, full 2-parameter Weibull, **uniform on a sub-interval $[a,b]\ne[0,1]$** (truncated — its support cuts off the posterior). → integrate numerically.
>
> **Special case:** uniform on the *full* range — $[0,1]$ for a probability — **is** $\text{Beta}(1,1)$, so it *is* conjugate and the Beta shortcut applies. Only the *truncated* uniform forces the integral.

---

## Gamma percentiles by hand (no computer) — the $\chi^2$ identity

For $\lambda\sim\text{Gamma}(\alpha,\beta)$ (rate parametrization):
$$2\beta\lambda \sim \chi^2_{2\alpha} \quad\Longrightarrow\quad \boxed{\lambda_p=\frac{\chi^2_{p,\,2\alpha}}{2\beta}}$$
where $\chi^2_{p,\nu}$ is the value with **lower-tail probability $p$** and $\nu=2\alpha$ degrees of freedom (read from a $\chi^2$ table; requires integer or half-integer $\alpha$ for table lookup).

- One-sided **lower** 90% limit: $p=0.10 \Rightarrow \lambda_L=\chi^2_{0.10,2\alpha}/(2\beta)$
- One-sided **upper** 90% limit: $p=0.90 \Rightarrow \lambda_U=\chi^2_{0.90,2\alpha}/(2\beta)$
- Two-sided **90% interval**: $\big[\chi^2_{0.05,2\alpha}/(2\beta),\ \chi^2_{0.95,2\alpha}/(2\beta)\big]$

---

## Interval mechanics — one-sided vs two-sided (read the verb!)

| Wording in problem | Means | Percentile(s) |
|---|---|---|
| "90% **lower** confidence limit" | one-sided, $P(\theta>\theta_L)=0.90$ | $p_{10}$ (10th) |
| "90% **upper** confidence limit" | one-sided, $P(\theta<\theta_U)=0.90$ | $p_{90}$ (90th) |
| "90% **bounds**" / "90% confidence **interval**" | two-sided | $[p_{05},\,p_{95}]$ |
| "median value" | — | $p_{50}$ |

> Same "90%", different quantity. The lower bound of a two-sided 90% interval ($p_{05}$) is **not** the one-sided 90% lower limit ($p_{10}$). Modarres mixes these — always parse the exact phrase.

---

## Python template (non-conjugate: lognormal / normal prior)

```python
import numpy as np
from scipy import integrate, stats, optimize

# --- data model ---
x, T = 2, 8760          # failures, total time on test

# --- PRIOR: pick ONE line ---
# lognormal from 90% bounds:
lo90, hi90 = 3e-4, 3e-2
mu  = np.log(np.sqrt(lo90*hi90))            # ln(median)
sig = np.log(np.sqrt(hi90/lo90))/1.645      # ln(EF)/1.645
prior = lambda l: stats.lognorm.pdf(l, s=sig, scale=np.exp(mu))
# normal from mean,std:   prior = lambda l: stats.norm.pdf(l, MEAN, STD)
# uniform on [a,b]:       prior = lambda l: 1.0/(b-a)   # constant; cancels in Z. Set LO,HI = a,b below.

# --- likelihood (Poisson counts / exponential TTF kernel) ---
like = lambda l: (l*T)**x * np.exp(-l*T)     # /x! cancels in normalization

# --- posterior, normalized ---
un = lambda l: like(l)*prior(l)
LO, HI = 1e-7, 1e-1                          # integration bounds (widen if needed)
Z  = integrate.quad(un, LO, HI)[0]
post = lambda l: un(l)/Z

# --- moments & percentiles ---
mean = integrate.quad(lambda l: l*post(l),   LO, HI)[0]
var  = integrate.quad(lambda l: l*l*post(l), LO, HI)[0] - mean**2
cdf  = lambda L: integrate.quad(post, LO, L)[0]
pct  = lambda p: optimize.brentq(lambda L: cdf(L)-p, LO, HI)

print(mean, np.sqrt(var))
print("two-sided 90%:", pct(0.05), pct(0.95))
print("one-sided lower 90%:", pct(0.10))
```

**Conjugate case needs no integration** — use the closed forms directly:
```python
from scipy import stats
# Gamma-Poisson:  a,b = a0+r, b0+T ;  g = stats.gamma(a, scale=1/b)
# Beta-Binomial:  a,b = a0+k, b0+n-k ; g = stats.beta(a, b)
# percentile:     g.ppf(p)   |   mean: g.mean()   |   var: g.var()
```

**Truncated-uniform prior on a reliability/probability $R\in[a,b]$** (e.g. "reliability is more than 0.9, uniform") with binomial data — non-conjugate, integrate over $[a,b]$:
```python
import numpy as np
from scipy import integrate, optimize

a, b = 0.9, 1.0                  # prior support: "more than 0.9" -> [0.9, 1]
n, k = 7, 1                      # n trials, k failures
s    = n - k                     # successes
like = lambda R: R**s * (1-R)**k # binomial kernel; C(n,k) AND prior height 1/(b-a) both cancel

Z    = integrate.quad(like, a, b)[0]
post = lambda R: like(R)/Z
mean = integrate.quad(lambda R: R*post(R), a, b)[0]
var  = integrate.quad(lambda R: R*R*post(R), a, b)[0] - mean**2
cdf  = lambda x: integrate.quad(post, a, x)[0]
pct  = lambda p: optimize.brentq(lambda x: cdf(x)-p, a, b)

print("mean R =", round(mean,4), " sd =", round(np.sqrt(var),4))   # 0.937, 0.0244
print("90% two-sided:", round(pct(0.05),4), round(pct(0.95),4))
```

---

## Worked micro-examples (compressed)

**A. Gamma–Poisson (3.27 style).** $\lambda_g=10^{-3}$, SD $=\lambda_g/2$. 10 units × 1500 h, 1 failure.
→ $\alpha_0=(\mu/\sigma)^2=4$, $\beta_0=\mu/\sigma^2=4000$. $T=15000$, $r=1$.
→ Posterior Gamma$(5,\,19000)$. Mean $=5/19000=2.63\times10^{-4}$. Var $=5/19000^2$.
→ Lower 90% limit $=\chi^2_{0.10,10}/(2\cdot19000)=4.865/38000=1.28\times10^{-4}$.

**B. Lognormal–Poisson (3.28 style).** Bounds $[3\times10^{-4},3\times10^{-2}]$ → median $3\times10^{-3}$, EF $=10$, $\mu=\ln(3\times10^{-3})$, $\sigma=\ln10/1.645\approx1.40$. Data: 2 failures in 8760 h.
→ Non-conjugate → integrate. Posterior median $\approx3.3\times10^{-4}$, two-sided 90% $\approx[1.2\times10^{-4},\,7.2\times10^{-4}]$, EF shrinks $10\to\sim2.5$.

**C. Beta–Binomial.** Prior $p\sim\text{Beta}(2,98)$ (mean $0.02$). Observe 3 failures in 50 demands.
→ Posterior Beta$(5,\,145)$. Mean $=5/150=0.033$. Bounds via `beta.ppf(0.05/0.95)`.

**D. Truncated-uniform + binomial (3.67 style).** "Reliability more than 0.9, uniform" → prior uniform on $[0.9,1]$, height $1/(1-0.9)=10$. Data: 7 tested, 1 fails (6 survive).
→ Non-conjugate (truncated). The height 10 cancels in normalization. Integrate $R^6(1-R)$ over $[0.9,1]$.
→ Posterior mean $\approx0.937$. Note: data alone wants $6/7\approx0.857$, but the prior floor at $0.9$ forbids that, so the posterior piles up just above $0.9$.

---

## Traps & sanity checks

**Traps**
- *Total time on test:* multiply units × hours for Poisson counts (the easy factor-of-10 to drop).
- *Prior ≠ likelihood:* "exponential"/"Weibull" usually names the **data** model; the **rate** still gets its own prior.
- *One- vs two-sided:* "limit" (one-sided) vs "bounds/interval" (two-sided) → different percentile.
- *Lognormal median is the **geometric** mean* of the bounds, never the arithmetic mean.
- *EF constant:* $1.645$ for 90% bounds; $1.960$ for 95% bounds.
- *Integration bounds:* if `quad` misbehaves, widen `[LO,HI]` to cover the posterior tails.
- *"Uniform/noninformative" + a range:* the range is the **support** $[a,b]$, not a distractor — set integration limits to $[a,b]$. Height $1/(b-a)$ cancels, so its only effect is truncation. Don't reflexively default to uniform on $[0,1]$.
- *Reliability vs unreliability:* if the parameter is reliability $R=P(\text{survive})$, the binomial kernel is $R^{\text{successes}}(1-R)^{\text{failures}}$ — successes raise $R$'s exponent. (In Beta terms, successes go to $a$, failures to $b$ — the *opposite* of the unreliability convention.) Match the count to the exponent, every time.

**Sanity checks (do these every time)**
- Posterior mean sits **between** prior mean and the raw MLE ($r/T$ or $k/n$).
- Posterior is **narrower** than the prior (more data ⇒ smaller EF/variance).
- Units: $\beta_0$ in a Gamma prior carries units of **time** ("equivalent prior exposure").

---

## Decision flow (when stuck)

```
What parameter?  rate λ  ───────────────► data = counts/times → Poisson/exponential
                 probability p ──────────► data = failures/demands → binomial

How is the prior given?
   mean + SD, rate ........... Gamma   → CONJUGATE → add (r to α, T to β)
   mean + SD, probability .... Beta    → CONJUGATE → add (k to a, n−k to b)
   90% bounds / EF ........... Lognormal → NOT conjugate → integrate
   "normal", mean+SD ......... Normal    → NOT conjugate → integrate
   "uniform" on full range ... Beta(1,1) → CONJUGATE → Beta shortcut
   "uniform" + a range [a,b] . Truncated uniform → NOT conjugate → integrate over [a,b]

What's asked?
   mean / variance ........... posterior moments
   "lower/upper limit" ....... one-sided percentile (p10 / p90)
   "bounds / interval" ....... two-sided [p05, p95]
   "median" .................. p50
```
